# Multi-Agent Workflow

Demonstrates a sequential multi-agent workflow where a **PlannerAgent** creates a story outline and a **WriterAgent** expands it into a full story.
Each agent runs as an independent Foundry persistent agent on its own thread.

In [ ]:
%pip install azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())  # loads .env from repo root

endpoint = os.environ["AZURE_FOUNDRY_ENDPOINT"]

client = AIProjectClient(endpoint=endpoint, credential=DefaultAzureCredential())
print("Client ready.")

In [ ]:
# Create two specialized agents
planner = client.agents.create_agent(
    model="gpt-4.1",
    name="PlannerAgent",
    instructions="You create concise story outlines with 3-4 bullet points.",
)

writer = client.agents.create_agent(
    model="gpt-4.1",
    name="WriterAgent",
    instructions="You write short, engaging stories (3-4 paragraphs) based on outlines.",
)

print(f"PlannerAgent: {planner.id}")
print(f"WriterAgent:  {writer.id}")

In [ ]:
# Step 1 — Planner creates the story outline
print("=== PlannerAgent: Creating Story Outline ===\n")

planner_thread = client.agents.create_thread()
client.agents.create_message(
    thread_id=planner_thread.id,
    role="user",
    content="Create an outline for a story about an AI that learns to paint.",
)

client.agents.create_and_process_run(thread_id=planner_thread.id, agent_id=planner.id)

planner_messages = client.agents.list_messages(thread_id=planner_thread.id)
outline = ""
for msg in planner_messages:
    if msg.role == "assistant":
        outline = msg.content[0].text.value
        break

print(outline)

In [ ]:
# Step 2 — Writer expands the outline into a full story
print("\n=== WriterAgent: Writing the Story ===\n")

writer_thread = client.agents.create_thread()
client.agents.create_message(
    thread_id=writer_thread.id,
    role="user",
    content=f"Write a story based on this outline:\n{outline}",
)

client.agents.create_and_process_run(thread_id=writer_thread.id, agent_id=writer.id)

writer_messages = client.agents.list_messages(thread_id=writer_thread.id)
for msg in writer_messages:
    if msg.role == "assistant":
        print(msg.content[0].text.value)
        break

In [ ]:
# Cleanup: delete both agents
client.agents.delete_agent(agent_id=planner.id)
client.agents.delete_agent(agent_id=writer.id)
print("Both agents deleted.")